<a href="https://colab.research.google.com/github/RafihaikalP/BigData26_A_2411532002_Rafi-Haikal-Pratama/blob/main/Praktikum4/BD_P04_2411532002_Rafi_Haikal_Pratama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Praktikum Big Data — Pertemuan 4

## K-1. Menyiapkan Lingkungan dan Data Contoh
Memasang pustaka, membaca `trips_bersih/`, dan mengganti nama kolom menjadi nama yang mudah dibaca. Sel pertama khusus Colab; sel berikutnya berlaku di mana saja.

> Bila muncul `ModuleNotFoundError`, jalankan ulang sel `pip install`, lalu **Runtime → Restart session**.

In [7]:
!pip install -q duckdb networkx shapely tinydb "deltalake>=1.0"

In [8]:
from google.colab import drive
drive.mount("/content/drive")

DIR_P3 = "/content/drive/MyDrive/BigData/Praktikum3"
DIR_SIMPAN = "/content/drive/MyDrive/BigData/Praktikum4"
TRIPS_BERSIH = f"{DIR_P3}/trips_bersih"   # hasil Praktikum 3 (K-10)

import zipfile

# Path ke file zip
zip_path = "/content/drive/MyDrive/BigData/Praktikum3/trips_bersih.zip"
# Path folder tujuan ekstraksi
extract_to = "/content/drive/MyDrive/BigData/Praktikum3"

# Proses ekstraksi
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_to)

print("Berhasil mengekstrak trips_bersih.zip!")

import os
os.makedirs(DIR_SIMPAN, exist_ok=True)
assert os.path.exists(TRIPS_BERSIH), "Folder trips_bersih/ tidak ada. Jalankan ulang P3/K-10 atau pakai K-1b."

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Berhasil mengekstrak trips_bersih.zip!


In [10]:
!pip install fastparquet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 33.8 MB/s eta 0:00:00


In [11]:
import os, time, json, sqlite3, random, shutil, glob
import numpy as np, pandas as pd

# Pengukur waktu (dari Praktikum 1/K-3, versi ringkas)
catatan = []
def ukur(label, fungsi):
    t0 = time.perf_counter()
    hasil = fungsi()
    detik = time.perf_counter() - t0
    catatan.append({"langkah": label, "detik": round(detik, 4)})
    print(f"[{label}] {detik:.4f} s")
    return hasil

df = ukur("baca_trips_bersih", lambda: pd.read_parquet(TRIPS_BERSIH, engine="fastparquet"))

# Ganti nama kolom asli data TLC menjadi nama yang lebih mudah dibaca
NAMA_BARU = {
    "tpep_pickup_datetime": "waktu_jemput",
    "PULocationID": "zona_jemput",
    "DOLocationID": "zona_tujuan",
    "borough_naik": "wilayah_jemput",
    "trip_distance": "jarak_mil",
    "total_amount": "total_bayar",
    "tip_amount": "tip",
    "nama_pembayaran": "metode_bayar",
}
df = df.rename(columns=NAMA_BARU)
df["wilayah_jemput"] = df["wilayah_jemput"].astype(str)
print("baris:", f"{len(df):,}", "| kolom:", df.shape[1])

# Contoh 300.000 baris untuk K-2 dan K-3 (agar tidak terlalu lama)
# Bila RAM habis pada K-2, ubah 300_000 menjadi 100_000
sample = df.sample(n=min(300_000, len(df)), random_state=42).reset_index(drop=True)
sample["id_perjalanan"] = sample.index + 1
sample["waktu_jemput"] = sample["waktu_jemput"].astype(str)

[baca_trips_bersih] 11.5381 s
baris: 2,986,910 | kolom: 18


### K-1b. Bila `trips_bersih/` Hilang (Jalur Cadangan)
Sel di bawah membuat **data tiruan** dengan kolom yang sama. Angka hasilnya berbeda dari data taksi nyata dan **harus dilaporkan sebagai data tiruan**.
Untuk memakainya: buka komentar baris terakhir, jalankan, lalu jalankan ulang sel K-1 (sel kode umum) dari atas.

In [ ]:
# Hanya dipakai bila trips_bersih/ Anda hilang: membuat data tiruan dengan kolom yang sama.
import numpy as np, pandas as pd, os, shutil

def bangkitkan_trips_sintetis(keluar, n=1_000_000, seed=42):
    rng = np.random.default_rng(seed)
    wilayah = ["Manhattan", "Brooklyn", "Queens", "Bronx", "Staten Island", "EWR"]
    zona_ids = np.arange(1, 264)
    wilayah_zona = rng.choice(wilayah, size=len(zona_ids), p=[0.35, 0.25, 0.22, 0.10, 0.05, 0.03])
    zona = pd.DataFrame({"LocationID": zona_ids, "borough_naik": wilayah_zona})
    pop = np.clip(rng.zipf(1.6, size=len(zona_ids)).astype(float), 1, 60)
    pop /= pop.sum()
    pu = rng.choice(zona_ids, size=n, p=pop)
    do = rng.choice(zona_ids, size=n, p=pop)
    awal = pd.Timestamp("2023-01-01")
    pickup = awal + pd.to_timedelta(rng.integers(0, 31 * 24 * 60, n), unit="m")
    durasi = rng.integers(2, 90, n)
    dist = np.round(rng.gamma(2.0, 1.6, n).clip(0.05, 60), 2)
    fare = np.round(2.5 + dist * 2.8 + rng.gamma(2.0, 1.0, n), 2)
    tip = np.round(rng.gamma(1.5, 1.2, n), 2)
    pay = rng.choice([1, 2, 3, 4], size=n, p=[0.7, 0.27, 0.02, 0.01])
    d = pd.DataFrame({
        "tpep_pickup_datetime": pickup,
        "tpep_dropoff_datetime": pickup + pd.to_timedelta(durasi, unit="m"),
        "passenger_count": rng.integers(1, 5, n).astype("float64"),
        "trip_distance": dist, "PULocationID": pu, "DOLocationID": do,
        "payment_type": pay, "fare_amount": fare, "tip_amount": tip,
        "total_amount": np.round(fare + tip + 3.0, 2),
        "durasi_menit": durasi.astype("float64"),
        "nama_pembayaran": pd.Series(pay).map({1: "Kartu kredit", 2: "Tunai", 3: "Gratis", 4: "Sengketa"}).values,
    })
    d = d.merge(zona.rename(columns={"LocationID": "PULocationID"}), on="PULocationID", how="left")
    d["suhu_c"] = np.round(rng.normal(3, 5, n), 1)
    d["hujan_mm"] = np.round(rng.gamma(0.4, 0.8, n), 2)
    if os.path.exists(keluar):
        shutil.rmtree(keluar)
    d.to_parquet(keluar, partition_cols=["borough_naik"], index=False)
    return keluar

# Buka komentar baris di bawah untuk memakainya, lalu jalankan ulang sel K-1 (sel kode umum):
# TRIPS_BERSIH = bangkitkan_trips_sintetis("/content/trips_bersih_sintetis")

## K-2. Tabel Biasa vs Key-value
Menyimpan perjalanan yang sama dalam dua cara, lalu mengajukan dua pertanyaan berbeda.


In [12]:
KOLOM = ["id_perjalanan", "waktu_jemput", "zona_jemput", "zona_tujuan",
         "wilayah_jemput", "jarak_mil", "total_bayar", "metode_bayar"]
data = sample[KOLOM].copy()

# Model 1 - relasional: tabel SQLite
if os.path.exists("uji.db"):
    os.remove("uji.db")
con = sqlite3.connect("uji.db")
ukur("relasional: muat tabel", lambda: data.to_sql("perjalanan", con, index=False))

# Model 2 - key-value: kamus (dict) dengan kunci "perjalanan:<id>", nilainya teks JSON
def bangun_kv():
    return {f"perjalanan:{r['id_perjalanan']}": json.dumps(r) for r in data.to_dict("records")}

kv = ukur("key-value: bangun dict", bangun_kv)
print("jumlah key:", len(kv))

[relasional: muat tabel] 1.4520 s
[key-value: bangun dict] 4.2014 s
jumlah key: 300000


In [13]:
# Pertanyaan A: ambil SATU perjalanan berdasarkan id (200 id acak)
random.seed(42)
id_uji = random.sample(range(1, len(data) + 1), 200)

def cari_sql():
    cur = con.cursor()
    return [cur.execute("SELECT * FROM perjalanan WHERE id_perjalanan = ?", (i,)).fetchone() for i in id_uji]

def cari_kv():
    return [json.loads(kv[f"perjalanan:{i}"]) for i in id_uji]

r1 = ukur("cari 200 perjalanan - SQL tanpa indeks", cari_sql)
con.execute("CREATE INDEX idx_id ON perjalanan(id_perjalanan)")   # buat indeks
r2 = ukur("cari 200 perjalanan - SQL dengan indeks", cari_sql)
r3 = ukur("cari 200 perjalanan - key-value", cari_kv)
print("hasil sama?", r1[0][0] == r2[0][0] == r3[0]["id_perjalanan"])

[cari 200 perjalanan - SQL tanpa indeks] 4.1811 s
[cari 200 perjalanan - SQL dengan indeks] 0.0037 s
[cari 200 perjalanan - key-value] 0.0011 s
hasil sama? True


In [14]:
# Pertanyaan B: hitung rata-rata total bayar per wilayah (melibatkan SEMUA data)
def hitung_sql():
    return con.execute("""SELECT wilayah_jemput, COUNT(*), ROUND(AVG(total_bayar), 2)
                          FROM perjalanan GROUP BY wilayah_jemput ORDER BY 2 DESC, 1""").fetchall()

def hitung_kv():
    total, jumlah = {}, {}
    for teks in kv.values():
        d = json.loads(teks)                  # setiap nilai harus dibuka dulu
        w = d["wilayah_jemput"]
        total[w] = total.get(w, 0) + d["total_bayar"]
        jumlah[w] = jumlah.get(w, 0) + 1
    return sorted([(w, jumlah[w], round(total[w] / jumlah[w], 2)) for w in jumlah],
                  key=lambda x: (-x[1], x[0]))

h1 = ukur("hitung per wilayah - SQL GROUP BY", hitung_sql)
h2 = ukur("hitung per wilayah - key-value (buka semua nilai)", hitung_kv)
print(h1[:3])
print("hasil sama?", h1 == h2)

[hitung per wilayah - SQL GROUP BY] 0.1820 s
[hitung per wilayah - key-value (buka semua nilai)] 1.2387 s
[('Manhattan', 267034, 22.7), ('Queens', 27288, 70.34), ('Unknown', 3787, 39.13)]
hasil sama? True


## K-3. Document: Bentuk Data yang Fleksibel
Menyimpan perjalanan sebagai dokumen JSON bersarang (embedding). Field `tip` sengaja hanya ada pada pembayaran kartu, mirip data TLC asli yang tidak mencatat tip tunai.

In [15]:
from tinydb import TinyDB, Query
from tinydb.storages import MemoryStorage

def ke_dokumen(r):
    dok = {
        "id_perjalanan": int(r.id_perjalanan),
        "waktu_jemput": r.waktu_jemput,
        "jemput": {"zona": int(r.zona_jemput), "wilayah": r.wilayah_jemput},
        "tujuan": {"zona": int(r.zona_tujuan)},
        "biaya": {"jarak_mil": float(r.jarak_mil),
                  "total": float(r.total_bayar),
                  "metode_bayar": r.metode_bayar},
    }
    # Tip hanya dicatat untuk pembayaran kartu -> field ini ada di sebagian dokumen saja
    if r.metode_bayar == "Kartu kredit":
        dok["biaya"]["tip"] = float(r.tip)
    return dok

db = TinyDB(storage=MemoryStorage)
tabel = db.table("perjalanan")
docs = [ke_dokumen(r) for r in sample.head(20_000).itertuples()]
tabel.insert_multiple(docs)
print("jumlah dokumen:", len(tabel))
print(json.dumps(tabel.all()[0], indent=2))

jumlah dokumen: 20000
{
  "id_perjalanan": 1,
  "waktu_jemput": "2023-01-16 10:40:39",
  "jemput": {
    "zona": 48,
    "wilayah": "Manhattan"
  },
  "tujuan": {
    "zona": 164
  },
  "biaya": {
    "jarak_mil": 1.55,
    "total": 19.32,
    "metode_bayar": "Kartu kredit",
    "tip": 3.22
  }
}


In [16]:
Q = Query()
print("jemput di Manhattan & total > 50 :", len(tabel.search((Q.jemput.wilayah == "Manhattan") & (Q.biaya.total > 50))))
print("dokumen yang punya field tip     :", len(tabel.search(Q.biaya.tip.exists())))
print("dokumen yang TIDAK punya tip     :", len(tabel.search(~Q.biaya.tip.exists())))

# Menambah field baru ("cuaca") pada satu dokumen - tanpa mengubah struktur apa pun
tabel.insert({"id_perjalanan": 999_999, "waktu_jemput": "2023-01-15 08:00:00",
              "jemput": {"zona": 161, "wilayah": "Manhattan"}, "tujuan": {"zona": 236},
              "biaya": {"jarak_mil": 1.2, "total": 14.5, "metode_bayar": "Kartu kredit", "tip": 3.0},
              "cuaca": {"hujan": True, "suhu_c": -2.0}})
print("dokumen yang punya field cuaca   :", len(tabel.search(Q.cuaca.exists())))
print("total dokumen sekarang           :", len(tabel))

jemput di Manhattan & total > 50 : 693
dokumen yang punya field tip     : 15906
dokumen yang TIDAK punya tip     : 4094
dokumen yang punya field cuaca   : 1
total dokumen sekarang           : 20001


## K-4. Penyimpanan per Kolom: DuckDB pada Parquet
Menjalankan SQL **langsung** pada folder Parquet berpartisi tanpa memuatnya ke tabel dulu, lalu membandingkannya dengan SQLite (penyimpanan per baris). *View* adalah "jendela" bernama di atas file Parquet; data tidak disalin.

In [17]:
import duckdb

GLOB = f"{TRIPS_BERSIH}/*/*.parquet"
con_d = duckdb.connect()

# VIEW = "jendela" bernama yang membaca Parquet langsung (tidak menyalin data).
# Di sinilah nama kolom asli TLC diberi nama yang mudah dibaca.
con_d.sql(f"""
    CREATE VIEW perjalanan AS
    SELECT tpep_pickup_datetime AS waktu_jemput,
           PULocationID         AS zona_jemput,
           DOLocationID         AS zona_tujuan,
           borough_naik         AS wilayah_jemput,
           trip_distance        AS jarak_mil,
           total_amount         AS total_bayar,
           nama_pembayaran      AS metode_bayar
    FROM read_parquet('{GLOB}', hive_partitioning=true)""")

SQL = """SELECT wilayah_jemput, COUNT(*) AS jumlah, ROUND(AVG(total_bayar), 2) AS rata_bayar
         FROM perjalanan GROUP BY wilayah_jemput ORDER BY jumlah DESC, wilayah_jemput"""
hasil_d = ukur("DuckDB - hitung per wilayah (langsung dari Parquet)", lambda: con_d.sql(SQL).df())
print(hasil_d)

[DuckDB - hitung per wilayah (langsung dari Parquet)] 0.2272 s
               wilayah_jemput   jumlah  rata_bayar
0                   Manhattan  2659609       22.72
1                      Queens   270315       70.46
2                     Unknown    37609       39.24
3                    Brooklyn    15724       32.72
4                       Bronx     3074       34.44
5  __HIVE_DEFAULT_PARTITION__      340       74.72
6               Staten Island      211       74.07
7                         EWR       28      106.41


In [18]:
# Pembanding: SQLite (menyimpan per baris) pada data yang sama
KOLOM_S = ["wilayah_jemput", "total_bayar", "jarak_mil", "zona_jemput", "zona_tujuan"]
if os.path.exists("baris.db"):
    os.remove("baris.db")
con_s = sqlite3.connect("baris.db")
ukur("SQLite - muat seluruh baris ke tabel", lambda: df[KOLOM_S].to_sql("perjalanan", con_s, index=False))
hasil_s = ukur("SQLite - hitung per wilayah", lambda: con_s.execute(
    """SELECT wilayah_jemput, COUNT(*), ROUND(AVG(total_bayar), 2)
       FROM perjalanan GROUP BY wilayah_jemput ORDER BY 2 DESC, 1""").fetchall())
print("hasil sama?", [tuple(r) for r in hasil_d.itertuples(index=False)] == [tuple(r) for r in hasil_s])

[SQLite - muat seluruh baris ke tabel] 9.5751 s
[SQLite - hitung per wilayah] 1.7270 s
hasil sama? False


In [19]:
# Filter pada kolom partisi: DuckDB hanya membuka folder wilayah yang diminta (partition pruning, P3)
SQL2 = """SELECT COUNT(*), ROUND(AVG(total_bayar), 2)
          FROM perjalanan WHERE wilayah_jemput = 'Staten Island'"""
print(ukur("DuckDB - hanya wilayah Staten Island", lambda: con_d.sql(SQL2).fetchall()))

ukuran_pq = sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(TRIPS_BERSIH) for f in fs) / 1024**2
print(f"Parquet ({df.shape[1]} kolom): {ukuran_pq:.1f} MB | SQLite ({len(KOLOM_S)} kolom): {os.path.getsize('baris.db') / 1024**2:.1f} MB")

[DuckDB - hanya wilayah Staten Island] 0.0419 s
[(211, 74.07)]
Parquet (18 kolom): 66.5 MB | SQLite (5 kolom): 114.2 MB


## K-5. Graph: Pertanyaan tentang Keterhubungan
Perjalanan diringkas menjadi arus antar zona. Setiap **zona = node**, setiap pasangan zona asal-tujuan = **edge berarah** dengan bobot jumlah perjalanan. Hasil graph disilangkan dengan SQL agar kita yakin keduanya menjawab hal yang sama.

In [20]:
import networkx as nx

# Satu baris = satu arus: berapa perjalanan dari zona jemput ke zona tujuan tertentu
arus = ukur("hitung arus antar zona", lambda: con_d.sql("""
    SELECT zona_jemput AS asal, zona_tujuan AS tujuan, COUNT(*) AS jumlah
    FROM perjalanan GROUP BY 1, 2""").df())

# Zona = node (titik), arus = edge (garis berarah, bobotnya jumlah perjalanan)
G = nx.DiGraph()
for r in arus.itertuples(index=False):
    G.add_edge(int(r.asal), int(r.tujuan), jumlah=int(r.jumlah))
print("node (zona):", G.number_of_nodes(), "| edge (arus):", G.number_of_edges())

# Lima arus terbesar. Urutan dibuat pasti dengan pengurut tambahan (jumlah, asal, tujuan) - lihat M.5
top_graf = sorted(G.edges(data=True), key=lambda e: (-e[2]["jumlah"], e[0], e[1]))[:5]
print("graph:", [(a, b, d["jumlah"]) for a, b, d in top_graf])
top_sql = arus.sort_values(["jumlah", "asal", "tujuan"], ascending=[False, True, True]).head(5)
print("SQL  :", list(top_sql.itertuples(index=False, name=None)))

# Lima zona yang paling banyak menjadi tujuan
populer = sorted(G.in_degree(weight="jumlah"), key=lambda kv: (-kv[1], kv[0]))[:5]
print("zona tujuan terpopuler:", populer)

[hitung arus antar zona] 0.5069 s
node (zona): 262 | edge (arus): 21583
graph: [(237, 236, 22065), (236, 237, 18810), (236, 236, 14110), (237, 237, 13753), (264, 264, 13562)]
SQL  : [(237, 236, 22065), (236, 237, 18810), (236, 236, 14110), (237, 237, 13753), (264, 264, 13562)]
zona tujuan terpopuler: [(236, 144209), (237, 130369), (161, 113991), (230, 87706), (170, 87110)]


In [21]:
MIN = 50                       # abaikan arus yang terlalu jarang (kurang dari 50 perjalanan)
zona_awal = populer[0][0]      # mulai dari zona tujuan terpopuler

# Pertanyaan: zona mana yang bisa dicapai dalam TEPAT 2 langkah dari zona_awal?
# (1) cara graph: telusuri tetangga dari tetangga
def dua_langkah_graph():
    hasil = set()
    for b, d1 in G[zona_awal].items():
        if d1["jumlah"] < MIN:
            continue
        for c, d2 in G[b].items():
            if d2["jumlah"] >= MIN and c != zona_awal:
                hasil.add(c)
    return hasil

# (2) cara SQL: harus menggabungkan tabel arus dengan dirinya sendiri (self-join)
con_d.register("arus", arus)
def dua_langkah_sql():
    return set(con_d.sql(f"""
        SELECT DISTINCT a2.tujuan
        FROM arus a1 JOIN arus a2 ON a1.tujuan = a2.asal
        WHERE a1.asal = {zona_awal} AND a1.jumlah >= {MIN}
          AND a2.jumlah >= {MIN} AND a2.tujuan <> {zona_awal}""").df()["tujuan"])

hg = ukur("2 langkah - graph (NetworkX)", dua_langkah_graph)
hs = ukur("2 langkah - SQL self-join (DuckDB)", dua_langkah_sql)
print("zona awal:", zona_awal, "| terjangkau (graph):", len(hg), "| (SQL):", len(hs), "| sama?", hg == hs)

# (3) Jumlah langkah bebas: "dalam paling banyak k langkah" - cukup ganti angka k
H = nx.DiGraph([(a, b) for a, b, d in G.edges(data=True) if d["jumlah"] >= MIN])
for k in (1, 2, 3, 4):
    n = len(nx.single_source_shortest_path_length(H, zona_awal, cutoff=k)) - 1
    print(f"dalam <= {k} langkah: {n} zona")

[2 langkah - graph (NetworkX)] 0.0065 s
[2 langkah - SQL self-join (DuckDB)] 0.0142 s
zona awal: 236 | terjangkau (graph): 214 | (SQL): 214 | sama? True
dalam <= 1 langkah: 74 zona
dalam <= 2 langkah: 214 zona
dalam <= 3 langkah: 214 zona
dalam <= 4 langkah: 214 zona


## K-6. Indeks Spasial
Mencari titik jemput di dalam 1.000 kotak kecil (sekitar 1 km x 1 km) dengan tiga cara: **loop Python**, **NumPy tanpa indeks spasial**, dan **indeks STRtree**. Cara pertama hanya dijalankan untuk 5 kotak karena terlalu lambat.

> **Peringatan:** titik di bawah ini **SINTETIS** (data TLC 2023 hanya berisi ID zona, tanpa koordinat). Angkanya menggambarkan cara kerja indeks, bukan pola perjalanan taksi sebenarnya. Tulis hal ini di laporan.

In [22]:
from shapely import STRtree, box, points

# CATATAN: data TLC 2023 hanya berisi ID zona, TANPA koordinat. Titik di bawah ini SINTETIS.
rng = np.random.default_rng(42)
N = 200_000
bujur = rng.normal(-73.97, 0.06, N).clip(-74.25, -73.70)    # longitude
lintang = rng.normal(40.74, 0.05, N).clip(40.50, 40.92)     # latitude
titik = points(bujur, lintang)

# 1.000 wilayah pencarian berupa kotak kecil (sekitar 1 km x 1 km)
M = 1000
cx = rng.uniform(-74.05, -73.85, M)
cy = rng.uniform(40.65, 40.85, M)
kotak = [box(x - 0.005, y - 0.005, x + 0.005, y + 0.005) for x, y in zip(cx, cy)]

In [23]:
# Cara 1: periksa titik satu per satu dengan loop Python (hanya 5 kotak, 1.000 akan sangat lama)
def cara_loop():
    return [sum(1 for p in titik if k.contains(p)) for k in kotak[:5]]

# Cara 2: bandingkan seluruh titik sekaligus dengan NumPy (tanpa indeks spasial), 1.000 kotak
def cara_numpy():
    hasil = []
    for x, y in zip(cx, cy):
        m = (bujur >= x - 0.005) & (bujur <= x + 0.005) & (lintang >= y - 0.005) & (lintang <= y + 0.005)
        hasil.append(int(m.sum()))
    return hasil

h_loop = ukur("cara 1 - loop Python, 5 kotak", cara_loop)
h_np = ukur("cara 2 - NumPy, 1000 kotak", cara_numpy)

# Cara 3: indeks spasial STRtree (sejenis R-tree)
pohon = ukur("bangun indeks STRtree", lambda: STRtree(titik))
h_idx = ukur("cara 3 - STRtree, 1000 kotak",
             lambda: [len(pohon.query(k, predicate="contains")) for k in kotak])

print("5 kotak pertama - loop:", h_loop, "| STRtree:", h_idx[:5])
print("hasil NumPy dan STRtree sama?", h_np == h_idx, "| loop dan STRtree sama (5 pertama)?", h_loop == h_idx[:5])
print("total titik ditemukan (1000 kotak):", sum(h_idx))

[cara 1 - loop Python, 5 kotak] 13.0254 s
[cara 2 - NumPy, 1000 kotak] 0.4990 s
[bangun indeks STRtree] 0.0560 s
[cara 3 - STRtree, 1000 kotak] 0.1832 s
5 kotak pertama - loop: [1012, 84, 228, 748, 509] | STRtree: [1012, 84, 228, 748, 509]
hasil NumPy dan STRtree sama? True | loop dan STRtree sama (5 pertama)? True
total titik ditemukan (1000 kotak): 417545


## K-7. Delta Lake: Versi, Time Travel, dan Schema Enforcement
Menjawab tiga pertanyaan diskusi Lakehouse dari Praktikum 3/K-8. Tabel Delta dibuat, ditambah data, dikoreksi dengan *update*, lalu versinya ditelusuri.

> Pada langkah penegakan skema akan muncul `SchemaMismatchError`. Itu **bukan kesalahan**, melainkan hasil yang diharapkan. Bila muncul `AttributeError ... 'file_uris'` jalankan `pip install -U deltalake`, lalu restart session.

In [25]:
import shutil
import os
from deltalake import DeltaTable, write_deltalake

KOLOM_DELTA = ["waktu_jemput", "zona_jemput", "zona_tujuan",
               "wilayah_jemput", "jarak_mil", "total_bayar", "metode_bayar"]

data_delta = df[KOLOM_DELTA].sample(n=min(200_000, len(df)), random_state=42).reset_index(drop=True)

# GANTI: Simpan di penyimpanan lokal Colab (/content/...) bukan langsung di /content/drive/...
PATH_DELTA_LOCAL = "/content/trips_delta"
PATH_DELTA_DRIVE = f"{DIR_SIMPAN}/trips_delta"

# Hapus folder lokal jika sudah ada
if os.path.exists(PATH_DELTA_LOCAL):
    shutil.rmtree(PATH_DELTA_LOCAL)

# Versi 0: menulis data awal (100.000 baris)
write_deltalake(PATH_DELTA_LOCAL, data_delta.iloc[:100_000], mode="overwrite")
dt = DeltaTable(PATH_DELTA_LOCAL)
print("versi:", dt.version(), "| baris:", len(dt.to_pandas()))

# Versi 1: menambah 50.000 baris
write_deltalake(PATH_DELTA_LOCAL, data_delta.iloc[100_000:150_000], mode="append")
dt = DeltaTable(PATH_DELTA_LOCAL)
print("versi:", dt.version(), "| baris:", len(dt.to_pandas()))

# Versi 2: koreksi data. Anggap tarif Tunai di wilayah Bronx perlu ditambah 1.0
sekarang = data_delta.iloc[:150_000]
n_koreksi = int(((sekarang["wilayah_jemput"] == "Bronx") & (sekarang["metode_bayar"] == "Tunai")).sum())
print("baris yang akan dikoreksi:", n_koreksi)

dt.update(predicate="wilayah_jemput = 'Bronx' AND metode_bayar = 'Tunai'",
          updates={"total_bayar": "total_bayar + 1.0"})
print("versi:", DeltaTable(PATH_DELTA_LOCAL).version())

# OPTIONAL: Salin hasil akhir dari VM lokal ke Google Drive Anda
if os.path.exists(PATH_DELTA_DRIVE):
    shutil.rmtree(PATH_DELTA_DRIVE)
shutil.copytree(PATH_DELTA_LOCAL, PATH_DELTA_DRIVE)
print("Berhasil menyalin tabel Delta ke Google Drive!")

versi: 0 | baris: 100000
versi: 1 | baris: 150000
baris yang akan dikoreksi: 20
versi: 2
Berhasil menyalin tabel Delta ke Google Drive!


In [26]:
# TIME TRAVEL: baca tabel sebagaimana keadaannya pada versi tertentu
v0 = DeltaTable(PATH_DELTA, version=0).to_pandas()
v1 = DeltaTable(PATH_DELTA, version=1).to_pandas()
terbaru = DeltaTable(PATH_DELTA).to_pandas()
print("baris v0:", len(v0), "| v1:", len(v1), "| terbaru:", len(terbaru))
selisih = terbaru["total_bayar"].sum() - v1["total_bayar"].sum()
print("selisih total_bayar (terbaru - v1):", round(selisih, 2), "| jumlah baris dikoreksi:", n_koreksi)

# Riwayat perubahan tabel
for h in DeltaTable(PATH_DELTA).history():
    print(h.get("version"), h.get("operation"))

# Log transaksi: satu file JSON per versi
print(sorted(os.listdir(f"{PATH_DELTA}/_delta_log")))

baris v0: 100000 | v1: 150000 | terbaru: 150000
selisih total_bayar (terbaru - v1): 20.0 | jumlah baris dikoreksi: 20
2 UPDATE
1 WRITE
0 WRITE
['00000000000000000000.json', '00000000000000000001.json', '00000000000000000002.json']


In [27]:
# PENEGAKAN SKEMA: menambah data dengan kolom asing harus DITOLAK, tabel tidak berubah
versi_sebelum_gagal = DeltaTable(PATH_DELTA).version()
tambahan = data_delta.iloc[150_000:150_010].copy()
tambahan["kolom_asing"] = 1
try:
    write_deltalake(PATH_DELTA, tambahan, mode="append")
    print("append kolom asing: BERHASIL (tidak diharapkan)")
except Exception as e:
    print("append kolom asing DITOLAK:", type(e).__name__)
versi_setelah_gagal = DeltaTable(PATH_DELTA).version()
print("versi sebelum percobaan gagal:", versi_sebelum_gagal, "| sesudah:", versi_setelah_gagal)

append kolom asing DITOLAK: SchemaMismatchError
versi sebelum percobaan gagal: 2 | sesudah: 2


## K-8. Small File Problem pada Tabel Delta
Meniru data yang masuk sedikit demi sedikit (menghasilkan banyak file kecil, seperti Praktikum 3/K-6), lalu merapikannya dengan **compaction**.

> **Peringatan tentang VACUUM.** Kode di bawah memakai `retention_hours=0` bersama `dry_run=True` **hanya agar Anda melihat cakupan VACUUM tanpa menghapus apa pun**. Jangan menjalankan VACUUM dengan retention 0 jam (`dry_run=False`) pada tabel sungguhan: file yang mungkin masih dibaca proses lain akan terhapus dan time travel ke versi lama terputus. Nilai bawaan (sekitar 7 hari) ada untuk melindungi hal itu.

In [29]:
# Tentukan jalur di penyimpanan lokal Colab
PATH_DELTA = "/content/trips_delta"

dt = DeltaTable(PATH_DELTA)
file_aktif_awal = len(dt.file_uris())
print("file data aktif sebelum:", file_aktif_awal, "| versi:", dt.version())

# 20 kali menambah data kecil (500 baris) - seperti data yang masuk sedikit demi sedikit
for i in range(20):
    potongan = data_delta.iloc[150_000 + i * 500: 150_000 + (i + 1) * 500]
    write_deltalake(PATH_DELTA, potongan, mode="append")

dt = DeltaTable(PATH_DELTA)
file_aktif_kecil = len(dt.file_uris())
print("file data aktif setelah 20 append kecil:", file_aktif_kecil, "| versi:", dt.version(), "| baris:", len(dt.to_pandas()))
t0 = time.perf_counter(); _ = DeltaTable(PATH_DELTA).to_pandas(); t_sebelum = time.perf_counter() - t0

# COMPACTION: gabungkan file kecil menjadi file yang lebih besar (dicatat sebagai versi baru)
hasil = dt.optimize.compact()
print("hasil compact:", {k: hasil[k] for k in ("numFilesAdded", "numFilesRemoved")})

dt = DeltaTable(PATH_DELTA)
file_aktif_compact = len(dt.file_uris())
t0 = time.perf_counter(); _ = DeltaTable(PATH_DELTA).to_pandas(); t_sesudah = time.perf_counter() - t0
print("file data aktif setelah compact:", file_aktif_compact, "| versi:", dt.version(), "| baris:", len(dt.to_pandas()))
print(f"waktu baca sebelum compact: {t_sebelum:.4f} s | sesudah: {t_sesudah:.4f} s")

# File lama TIDAK langsung terhapus (masih dibutuhkan untuk time travel). VACUUM yang menghapusnya.
file_di_disk = len(glob.glob(f"{PATH_DELTA}/*.parquet"))
print("file parquet di disk:", file_di_disk)
kandidat = dt.vacuum(retention_hours=0, enforce_retention_duration=False, dry_run=True) # dry_run = hanya melaporkan
print("file yang akan dihapus VACUUM (dry run):", len(kandidat))

file data aktif sebelum: 1 | versi: 2
file data aktif setelah 20 append kecil: 21 | versi: 22 | baris: 160000
hasil compact: {'numFilesAdded': 1, 'numFilesRemoved': 21}
file data aktif setelah compact: 1 | versi: 23 | baris: 160000
waktu baca sebelum compact: 0.3805 s | sesudah: 0.1292 s
file parquet di disk: 24
file yang akan dihapus VACUUM (dry run): 23


## K-9. Merangkum dan Memilih
Menyimpan semua angka waktu ke `benchmark_nosql.csv`, lalu mengisi tabel rekap dengan angka Anda sendiri.

In [30]:
rekap = pd.DataFrame(catatan)
rekap.to_csv(f"{DIR_SIMPAN}/benchmark_nosql.csv", index=False)
print(rekap.to_string(index=False))

                                            langkah   detik
                                  baca_trips_bersih 11.5381
                             relasional: muat tabel  1.4520
                             key-value: bangun dict  4.2014
             cari 200 perjalanan - SQL tanpa indeks  4.1811
            cari 200 perjalanan - SQL dengan indeks  0.0037
                    cari 200 perjalanan - key-value  0.0011
                  hitung per wilayah - SQL GROUP BY  0.1820
  hitung per wilayah - key-value (buka semua nilai)  1.2387
DuckDB - hitung per wilayah (langsung dari Parquet)  0.2272
               SQLite - muat seluruh baris ke tabel  9.5751
                        SQLite - hitung per wilayah  1.7270
               DuckDB - hanya wilayah Staten Island  0.0419
                             hitung arus antar zona  0.5069
                       2 langkah - graph (NetworkX)  0.0065
                 2 langkah - SQL self-join (DuckDB)  0.0142
                      cara 1 - loop Pyth

In [31]:
# Tabel rekap K-9 dengan angka dari ukuran Anda sendiri (diisi otomatis dari daftar `catatan`)
def w(label):
    return next(c["detik"] for c in reversed(catatan) if c["langkah"] == label)

def x(a, b):
    return a / b if b else float("nan")

loop_1000 = w("cara 1 - loop Python, 5 kotak") / 5 * 1000          # perkiraan, tidak dijalankan
strtree_total = w("bangun indeks STRtree") + w("cara 3 - STRtree, 1000 kotak")

rekap_k9 = pd.DataFrame([
    ["Ambil satu data lewat id", "Key-value; tabel dengan indeks", "Tabel tanpa indeks",
     f"tanpa indeks {w('cari 200 perjalanan - SQL tanpa indeks'):.4f} s | berindeks {w('cari 200 perjalanan - SQL dengan indeks'):.4f} s | key-value {w('cari 200 perjalanan - key-value'):.4f} s "
     f"(tanpa indeks = {x(w('cari 200 perjalanan - SQL tanpa indeks'), w('cari 200 perjalanan - SQL dengan indeks')):.0f}x lebih lambat dari berindeks)"],
    ["Menghitung atau merangkum banyak baris", "Per kolom (DuckDB pada Parquet)", "Key-value (buka semua nilai)",
     f"DuckDB {w('DuckDB - hitung per wilayah (langsung dari Parquet)'):.4f} s | SQLite {w('SQLite - hitung per wilayah'):.4f} s "
     f"({x(w('SQLite - hitung per wilayah'), w('DuckDB - hitung per wilayah (langsung dari Parquet)')):.1f}x) | SQL GROUP BY {w('hitung per wilayah - SQL GROUP BY'):.4f} s vs key-value {w('hitung per wilayah - key-value (buka semua nilai)'):.4f} s "
     f"({x(w('hitung per wilayah - key-value (buka semua nilai)'), w('hitung per wilayah - SQL GROUP BY')):.1f}x)"],
    ["Data dengan field opsional atau berubah-ubah", "Document", "Tabel (kolom NULL, migrasi)", "(kualitatif)"],
    ["Keterhubungan dengan jumlah langkah bebas", "Graph", "SQL (satu JOIN per langkah)",
     f"2 langkah: graph {w('2 langkah - graph (NetworkX)'):.4f} s | SQL self-join {w('2 langkah - SQL self-join (DuckDB)'):.4f} s (bandingkan angkanya; keunggulan graph terutama pada cara bertanya: k = 1..4 cukup ganti angka)"],
    ["Mencari lokasi dalam wilayah", "Spasial dengan indeks", "Memeriksa semua titik",
     f"STRtree {strtree_total:.4f} s | NumPy {w('cara 2 - NumPy, 1000 kotak'):.4f} s ({x(w('cara 2 - NumPy, 1000 kotak'), strtree_total):.1f}x) | loop Python ~{loop_1000:.0f} s (perkiraan; {x(loop_1000, strtree_total):.0f}x)"],
    ["Perubahan yang harus aman dan bisa dibatalkan", "Tabel Delta", "Folder Parquet biasa", "(kualitatif)"],
], columns=["Kebutuhan", "Cara yang unggul", "Cara yang lemah", "Angka Anda"])

pd.set_option("display.max_colwidth", None)
display(rekap_k9)

,Kebutuhan,Cara yang unggul,Cara yang lemah,Angka Anda
0,Ambil satu data lewat id,Key-value; tabel dengan indeks,Tabel tanpa indeks,tanpa indeks 4.1811 s | berindeks 0.0037 s | key-value 0.0011 s (tanpa indeks = 1130x lebih lambat dari berindeks)
1,Menghitung atau merangkum banyak baris,Per kolom (DuckDB pada Parquet),Key-value (buka semua nilai),DuckDB 0.2272 s | SQLite 1.7270 s (7.6x) | SQL GROUP BY 0.1820 s vs key-value 1.2387 s (6.8x)
2,Data dengan field opsional atau berubah-ubah,Document,"Tabel (kolom NULL, migrasi)",(kualitatif)
3,Keterhubungan dengan jumlah langkah bebas,Graph,SQL (satu JOIN per langkah),2 langkah: graph 0.0065 s | SQL self-join 0.0142 s (bandingkan angkanya; keunggulan graph terutama pada cara bertanya: k = 1..4 cukup ganti angka)
4,Mencari lokasi dalam wilayah,Spasial dengan indeks,Memeriksa semua titik,STRtree 0.2392 s | NumPy 0.4990 s (2.1x) | loop Python ~2605 s (perkiraan; 10891x)
5,Perubahan yang harus aman dan bisa dibatalkan,Tabel Delta,Folder Parquet biasa,(kualitatif)


# Q. Latihan


### Latihan 1: Pencarian dengan 2.000 id acak (tabel berindeks vs key-value)
Hitung waktu rata-rata **per pencarian** dalam mikrodetik untuk masing-masing.

In [32]:
random.seed(7)
id_uji2 = random.sample(range(1, len(data) + 1), min(2000, len(data)))
n2 = len(id_uji2)

def cari_sql_2():
    cur = con.cursor()
    return [cur.execute("SELECT * FROM perjalanan WHERE id_perjalanan = ?", (i,)).fetchone() for i in id_uji2]

def cari_kv_2():
    return [json.loads(kv[f"perjalanan:{i}"]) for i in id_uji2]

ukur(f"L1 - {n2} pencarian SQL berindeks", cari_sql_2)
t_sql = catatan[-1]["detik"]
ukur(f"L1 - {n2} pencarian key-value", cari_kv_2)
t_kv = catatan[-1]["detik"]
print(f"rata-rata per pencarian | SQL berindeks: {t_sql / n2 * 1e6:.2f} mikrodetik | key-value: {t_kv / n2 * 1e6:.2f} mikrodetik")
print(f"rasio SQL berindeks / key-value: {t_sql / t_kv:.2f}x")

[L1 - 2000 pencarian SQL berindeks] 0.0346 s
[L1 - 2000 pencarian key-value] 0.0115 s
rata-rata per pencarian | SQL berindeks: 17.30 mikrodetik | key-value: 5.75 mikrodetik
rasio SQL berindeks / key-value: 3.01x


### Latihan 2: Hitung dokumen (TinyDB) dan SQL untuk zona tujuan tertentu & metode bayar "Tunai"
Pada K-3 hanya **20.000 baris pertama** yang dimuat ke TinyDB, sedangkan tabel SQL K-2 berisi 300.000 baris. Cek apakah angkanya sama, dan jika tidak, apa sebabnya.

In [33]:
# Pilih zona tujuan yang paling sering muncul pada 20.000 dokumen pertama
sub = sample.head(20_000)
ZONA_UJI = int(sub["zona_tujuan"].mode().iloc[0])
print("zona tujuan yang dipakai:", ZONA_UJI)

n_doc = len(tabel.search((Q.tujuan.zona == ZONA_UJI) & (Q.biaya.metode_bayar == "Tunai")))

sql_semua = con.execute("""SELECT COUNT(*) FROM perjalanan
                           WHERE zona_tujuan = ? AND metode_bayar = 'Tunai'""", (ZONA_UJI,)).fetchone()[0]
sql_20rb = con.execute("""SELECT COUNT(*) FROM perjalanan
                          WHERE zona_tujuan = ? AND metode_bayar = 'Tunai' AND id_perjalanan <= 20000""",
                       (ZONA_UJI,)).fetchone()[0]

print("TinyDB (20.000 dokumen)             :", n_doc)
print("SQL pada seluruh 300.000 baris      :", sql_semua)
print("SQL pada id <= 20.000 (sama cakupan):", sql_20rb)
print("TinyDB == SQL (seluruh)?", n_doc == sql_semua, "| TinyDB == SQL (id <= 20.000)?", n_doc == sql_20rb)

zona tujuan yang dipakai: 236
TinyDB (20.000 dokumen)             : 155
SQL pada seluruh 300.000 baris      : 2074
SQL pada id <= 20.000 (sama cakupan): 155
TinyDB == SQL (seluruh)? False | TinyDB == SQL (id <= 20.000)? True


### Latihan 3: DuckDB `SUM` satu kolom vs beberapa kolom (column pruning)

In [34]:
def ulang(sql, n=7):
    ws = []
    for _ in range(n):
        t0 = time.perf_counter(); con_d.sql(sql).fetchall(); ws.append(time.perf_counter() - t0)
    return min(ws), float(np.median(ws))

sql_satu = "SELECT SUM(total_bayar) FROM perjalanan"
sql_banyak = "SELECT SUM(total_bayar), SUM(jarak_mil), SUM(zona_jemput), SUM(zona_tujuan) FROM perjalanan"

ulang(sql_satu, 1); ulang(sql_banyak, 1)       # pemanasan (cache file)
m1, md1 = ulang(sql_satu)
m4, md4 = ulang(sql_banyak)
print(f"SUM 1 kolom   : min {m1:.4f} s | median {md1:.4f} s")
print(f"SUM 4 kolom   : min {m4:.4f} s | median {md4:.4f} s")
print(f"4 kolom / 1 kolom (median): {md4 / md1:.2f}x")

SUM 1 kolom   : min 0.2386 s | median 0.2515 s
SUM 4 kolom   : min 0.2956 s | median 0.5045 s
4 kolom / 1 kolom (median): 2.01x


### Latihan 4: Lima zona dengan jumlah zona tujuan berbeda terbanyak (graph out-degree vs SQL `COUNT(DISTINCT)`)

In [35]:
top_out = sorted(G.out_degree(), key=lambda kv_: (-kv_[1], kv_[0]))[:5]
print("graph (out-degree):", top_out)

top_sql_out = con_d.sql("""
    SELECT zona_jemput AS zona, COUNT(DISTINCT zona_tujuan) AS tujuan_berbeda
    FROM perjalanan GROUP BY 1 ORDER BY 2 DESC, 1 LIMIT 5""").fetchall()
print("SQL (COUNT DISTINCT):", top_sql_out)
print("sama?", [tuple(a) for a in top_out] == [tuple(b) for b in top_sql_out])

graph (out-degree): [(132, 260), (138, 254), (264, 241), (230, 234), (140, 233)]
SQL (COUNT DISTINCT): [(132, 260), (138, 254), (264, 241), (230, 234), (140, 233)]
sama? True


### Latihan 5: Ulangi K-6 dengan sisi kotak 0,02 derajat dan 0,001 derajat
Bagaimana perbandingan STRtree terhadap NumPy berubah? Mengapa kotak yang lebih kecil lebih menguntungkan indeks?

In [36]:
def uji_sisi(sisi):
    h = sisi / 2
    kotak_s = [box(x - h, y - h, x + h, y + h) for x, y in zip(cx, cy)]

    t0 = time.perf_counter()
    hasil_np = [int(((bujur >= x - h) & (bujur <= x + h) & (lintang >= y - h) & (lintang <= y + h)).sum())
                for x, y in zip(cx, cy)]
    t_np = time.perf_counter() - t0

    t0 = time.perf_counter(); pohon_s = STRtree(titik); t_bangun = time.perf_counter() - t0
    t0 = time.perf_counter()
    hasil_ix = [len(pohon_s.query(k, predicate="contains")) for k in kotak_s]
    t_cari = time.perf_counter() - t0

    return {"sisi (derajat)": sisi, "NumPy (s)": round(t_np, 4), "bangun indeks (s)": round(t_bangun, 4),
            "cari STRtree (s)": round(t_cari, 4),
            "NumPy / STRtree cari saja": round(t_np / t_cari, 2),
            "NumPy / (bangun+cari)": round(t_np / (t_bangun + t_cari), 2),
            "hasil sama?": hasil_np == hasil_ix, "titik ditemukan": sum(hasil_ix)}

tabel_sisi = pd.DataFrame([uji_sisi(s) for s in (0.02, 0.01, 0.001)])
display(tabel_sisi)

,sisi (derajat),NumPy (s),bangun indeks (s),cari STRtree (s),NumPy / STRtree cari saja,NumPy / (bangun+cari),hasil sama?,titik ditemukan
0,0.020,0.5887,0.0796,0.9058,0.65,0.60,True,1666631
1,0.010,0.6069,0.0663,0.2587,2.35,1.87,True,417545
2,0.001,0.6008,0.0524,0.0108,55.60,9.51,True,4149


### Latihan 6: Time travel vs `restore`
Jalankan sel ini **setelah K-8**. Jelaskan perbedaan time travel (membaca versi lama) dengan restore (mengembalikan tabel ke versi lama), dan mengapa restore dicatat sebagai versi baru, bukan menghapus riwayat.

In [37]:
dt = DeltaTable(PATH_DELTA)
print("versi sebelum restore:", dt.version(), "| baris:", len(dt.to_pandas()))
dt.restore(1)      # kembalikan tabel ke keadaan versi 1 (dicatat sebagai versi baru)
dt = DeltaTable(PATH_DELTA)
print("versi setelah restore:", dt.version(), "| baris:", len(dt.to_pandas()))
print([h.get("operation") for h in dt.history()][:3])

versi sebelum restore: 23 | baris: 160000
versi setelah restore: 24 | baris: 150000
['RESTORE', 'OPTIMIZE', 'WRITE']


In [38]:
# Tambahan: keadaan sebelum restore tetap bisa dibaca lewat time travel (riwayat tidak terhapus)
v_sebelum_restore = dt.version() - 1
print(f"time travel ke versi {v_sebelum_restore} (sebelum restore): baris =", len(DeltaTable(PATH_DELTA, version=v_sebelum_restore).to_pandas()))

time travel ke versi 23 (sebelum restore): baris = 160000


In [39]:
# Simpan ulang benchmark_nosql.csv (kini termasuk waktu dari latihan)
rekap = pd.DataFrame(catatan)
rekap.to_csv(f"{DIR_SIMPAN}/benchmark_nosql.csv", index=False)
print("benchmark_nosql.csv tersimpan:", len(rekap), "baris ->", f"{DIR_SIMPAN}/benchmark_nosql.csv")

benchmark_nosql.csv tersimpan: 21 baris -> /content/drive/MyDrive/BigData/Praktikum4/benchmark_nosql.csv


# O. Analisis Hasil
**Aturan: satu klaim, satu angka, satu alasan.** Sel di bawah mengumpulkan angka dari hasil ukur Anda sendiri; jawaban ditulis di sel Markdown sesudahnya.

In [40]:
# Bantuan angka untuk Analisis Hasil (diambil dari hasil ukur Anda sendiri)
print("1. Pencarian satu data")
print(f"   tanpa indeks / berindeks : {x(w('cari 200 perjalanan - SQL tanpa indeks'), w('cari 200 perjalanan - SQL dengan indeks')):.1f}x")
print(f"   berindeks / key-value    : {x(w('cari 200 perjalanan - SQL dengan indeks'), w('cari 200 perjalanan - key-value')):.2f}x")
print("2. Menghitung: key-value / SQL GROUP BY :",
      f"{x(w('hitung per wilayah - key-value (buka semua nilai)'), w('hitung per wilayah - SQL GROUP BY')):.1f}x")
print("4. Per kolom: SQLite / DuckDB           :",
      f"{x(w('SQLite - hitung per wilayah'), w('DuckDB - hitung per wilayah (langsung dari Parquet)')):.1f}x",
      f"(SQLite masih perlu muat tabel {w('SQLite - muat seluruh baris ke tabel'):.2f} s)")
print("5. Dua langkah: graph", w("2 langkah - graph (NetworkX)"), "s | SQL", w("2 langkah - SQL self-join (DuckDB)"), "s")
print("   zona terjangkau k=1..4 : lihat keluaran K-5")
print("6. Spasial: loop(1000, perkiraan) / STRtree total :", f"{x(loop_1000, strtree_total):.0f}x")
print("            NumPy / STRtree total                 :", f"{x(w('cara 2 - NumPy, 1000 kotak'), strtree_total):.2f}x")
print("            NumPy / STRtree cari saja             :", f"{x(w('cara 2 - NumPy, 1000 kotak'), w('cara 3 - STRtree, 1000 kotak')):.2f}x")
print("7. Delta: baris v0/v1/terbaru =", len(v0), "/", len(v1), "/", len(terbaru),
      "| selisih total_bayar =", round(selisih, 2), "vs baris dikoreksi =", n_koreksi)
print("          versi sebelum/sesudah penulisan gagal =", versi_sebelum_gagal, "/", versi_setelah_gagal)
print("          file aktif setelah compact =", file_aktif_compact, "| file parquet di disk =", file_di_disk)

1. Pencarian satu data
   tanpa indeks / berindeks : 1130.0x
   berindeks / key-value    : 3.36x
2. Menghitung: key-value / SQL GROUP BY : 6.8x
4. Per kolom: SQLite / DuckDB           : 7.6x (SQLite masih perlu muat tabel 9.58 s)
5. Dua langkah: graph 0.0065 s | SQL 0.0142 s
   zona terjangkau k=1..4 : lihat keluaran K-5
6. Spasial: loop(1000, perkiraan) / STRtree total : 10891x
            NumPy / STRtree total                 : 2.09x
            NumPy / STRtree cari saja             : 2.72x
7. Delta: baris v0/v1/terbaru = 100000 / 150000 / 150000 | selisih total_bayar = 20.0 vs baris dikoreksi = 20
          versi sebelum/sesudah penulisan gagal = 2 / 2
          file aktif setelah compact = 1 | file parquet di disk = 24
